# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Big Data** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Examples on Machine Learning: Alternating Least Squares (ALS)** </center>
---
**Profesor**: Pablo Camarillo Ramirez

# Create SparkSession

In [1]:
from spark_utils import SparkUtils

su = SparkUtils("ML: ALS", 
                "spark://spark-master:7077")
su.spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/23 01:19:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Example 1: Songs recommednation

In [2]:
# Sample user-song interaction data
data = [(1, 1, 4),
        (1, 2, 5),
        (1, 5, 5),
        (2, 2, 3),
        (2, 3, 4),
        (2, 4, 3),
        (3, 1, 2),
        (3, 3, 5),
        (3, 5, 1)]
  
# Define schema for the DataFrame
schema = SparkUtils.generate_schema([("user_id", "int"), ("song_id", "int"), ("rating", "int")])

# Create DataFrame for interactions
interactions_df = su.spark.createDataFrame(data, schema)
interactions_df.show()

+-------+-------+------+
|user_id|song_id|rating|
+-------+-------+------+
|      1|      1|     4|
|      1|      2|     5|
|      1|      5|     5|
|      2|      2|     3|
|      2|      3|     4|
|      2|      4|     3|
|      3|      1|     2|
|      3|      3|     5|
|      3|      5|     1|
+-------+-------+------+



In [3]:
print(f"Number of items o canciones (n):{interactions_df.groupBy('song_id').count().count()}")
print(f"Number of users (m):{interactions_df.groupBy('user_id').count().count()}")

Number of items o canciones (n):5
Number of users (m):3


In [4]:
from pyspark.ml.recommendation import ALS

als = ALS(
    userCol="user_id", 
    itemCol="song_id", 
    ratingCol="rating", 
    maxIter=10, 
    regParam=0.1, 
    rank=5, # Controls the dimensionality of the latent vector space for 
            # users and items.
    coldStartStrategy="drop"  # Avoids NaN predictions
)

In [5]:
model = als.fit(interactions_df)
print("Recommendation system generated successfully")

Recommendation system generated successfully


In [6]:
# Generate recommendations for each user
user_recommendations = model.recommendForAllUsers(numItems=3)

# Show recommendations
user_recommendations.show(truncate=False)

[Stage 91:======================================================>(99 + 1) / 100]

+-------+------------------------------------------------+
|user_id|recommendations                                 |
+-------+------------------------------------------------+
|1      |[{2, 4.9494977}, {5, 4.855001}, {1, 3.9374762}] |
|2      |[{3, 3.9434912}, {2, 2.9666052}, {4, 2.9092283}]|
|3      |[{3, 4.832548}, {4, 3.2140925}, {2, 2.4086003}] |
+-------+------------------------------------------------+



In [7]:
songs = [
    (1, "song a"),
    (2, "song b"),
    (3, "song c"),
    (4, "song d"),
    (5, "song e")]

songs_schema = SparkUtils.generate_schema([("song_id", "int"), ("title", "string")])
songs_df = su.spark.createDataFrame(songs, songs_schema)

In [8]:
from pyspark.sql.functions import explode

# Explode recommendations for easier reading
recommendations = user_recommendations.select("user_id", explode("recommendations").alias("rec"))
recommendations = recommendations.join(songs_df, recommendations.rec.song_id == songs_df.song_id).select("user_id", "title", "rec.rating")

# Show user-song recommendations with titles
recommendations.show(truncate=False)

+-------+------+---------+
|user_id|title |rating   |
+-------+------+---------+
|1      |song b|4.9494977|
|1      |song e|4.855001 |
|1      |song a|3.9374762|
|2      |song c|3.9434912|
|2      |song b|2.9666052|
|2      |song d|2.9092283|
|3      |song c|4.832548 |
|3      |song d|3.2140925|
|3      |song b|2.4086003|
+-------+------+---------+



In [9]:
predictions = model.transform(interactions_df)
predictions.show(truncate=False)

+-------+-------+------+----------+
|user_id|song_id|rating|prediction|
+-------+-------+------+----------+
|1      |1      |4     |3.9374762 |
|1      |2      |5     |4.9494977 |
|1      |5      |5     |4.855001  |
|2      |2      |3     |2.9666052 |
|3      |1      |2     |1.9643906 |
|3      |3      |5     |4.832548  |
|3      |5      |1     |1.0484891 |
|2      |3      |4     |3.9434912 |
|2      |4      |3     |2.9092283 |
+-------+-------+------+----------+



In [10]:
# Evaluate the Recommendation System
from pyspark.ml.evaluation import RegressionEvaluator
# Set up evaluator to compute RMSE
evaluator = RegressionEvaluator(
    metricName="rmse", 
    labelCol="rating", 
    predictionCol="prediction"
)

# Calculate RMSE
rmse = evaluator.evaluate(predictions)
print(f"Root-mean-square error (RMSE) = {rmse}")

Root-mean-square error (RMSE) = 0.0892510365653541


# Lab 12: Building a Recommendation System with ALS 

In [11]:
movies_ratings_path = "/opt/spark/work-dir/data/ml/als"

movies_ratings_schema = SparkUtils.generate_schema([("user_id", "int"), ("movie_id", "int"), ("rating", "int"),("timestamp", "int")])

# Source https://github.com/databricks/Spark-The-Definitive-Guide/blob/master/data/sample_movielens_ratings.txt
movies_ratings_df = su.spark.read \
                    .option("header", "false") \
                    .option("delimiter", "::") \
                    .schema(movies_ratings_schema) \
                    .csv(movies_ratings_path)

movies_ratings_df.printSchema()
movies_ratings_df.show(n=3)

root
 |-- user_id: integer (nullable = true)
 |-- movie_id: integer (nullable = true)
 |-- rating: integer (nullable = true)
 |-- timestamp: integer (nullable = true)

+-------+--------+------+----------+
|user_id|movie_id|rating| timestamp|
+-------+--------+------+----------+
|      0|       2|     3|1424380312|
|      0|       3|     1|1424380312|
|      0|       5|     2|1424380312|
+-------+--------+------+----------+
only showing top 3 rows


In [12]:
print(f"Number of items o movies (n):{movies_ratings_df.groupBy('movie_id').count().count()}")
print(f"Number of users (m):{movies_ratings_df.groupBy('user_id').count().count()}")

Number of items o movies (n):100
Number of users (m):30


## Create & Train the ML Model

In [21]:
from pyspark.ml.recommendation import ALS

als = ALS(
    userCol="user_id", 
    itemCol="movie_id", 
    ratingCol="rating", 
    maxIter=20, 
    regParam=0.1, 
    rank=5, # Controls the dimensionality of the latent vector space for 
            # users and items.
    coldStartStrategy="drop"  # Avoids NaN predictions
)

In [22]:
als_model = als.fit(movies_ratings_df)
print("Recommendation system generated successfully")

Recommendation system generated successfully


## Persist the model

In [23]:
model_path = "/opt/spark/work-dir/data/mlmodels/als/als1"
als_model.write().overwrite().save(model_path)

In [24]:
!ls -lah /opt/spark/work-dir/data/mlmodels/als/als1/

total 0
drwxr-xr-x 1 root root 4.0K Apr 23 01:22 .
drwxr-xr-x 1 root root 4.0K Apr 23 01:21 ..
drwxr-xr-x 1 root root 4.0K Apr 23 01:22 itemFactors
drwxr-xr-x 1 root root 4.0K Apr 23 01:22 metadata
drwxr-xr-x 1 root root 4.0K Apr 23 01:22 userFactors


## Predictions

In [25]:
# Generate recommendations for each user
user_recommendations = model.recommendForAllUsers(numItems=30)

# Show recommendations
user_recommendations.show(truncate=False)

import pandas as pd
recs_pd = user_recommendations.toPandas()
print(recs_pd)

+-------+-------------------------------------------------------------------------------+
|user_id|recommendations                                                                |
+-------+-------------------------------------------------------------------------------+
|1      |[{2, 4.9494977}, {5, 4.855001}, {1, 3.9374762}, {4, 2.9502318}, {3, 2.9363406}]|
|2      |[{3, 3.9434912}, {2, 2.9666052}, {4, 2.9092283}, {1, 2.3909798}, {5, 2.066902}]|
|3      |[{3, 4.832548}, {4, 3.2140925}, {2, 2.4086003}, {1, 1.9643906}, {5, 1.0484891}]|
+-------+-------------------------------------------------------------------------------+



[Stage 812:=========================================>            (77 + 1) / 100]

   user_id                                    recommendations
0        1  [(2, 4.949497699737549), (5, 4.855000972747803...
1        2  [(3, 3.943491220474243), (2, 2.966605186462402...
2        3  [(3, 4.832548141479492), (4, 3.214092493057251...


In [26]:
predictions = als_model.transform(movies_ratings_df)
predictions.show(truncate=False)

+-------+--------+------+----------+----------+
|user_id|movie_id|rating|timestamp |prediction|
+-------+--------+------+----------+----------+
|22     |0       |1     |1424380312|0.97997004|
|22     |3       |2     |1424380312|1.6270051 |
|22     |5       |2     |1424380312|2.1968977 |
|22     |6       |2     |1424380312|2.2761478 |
|22     |9       |1     |1424380312|1.7074243 |
|22     |10      |1     |1424380312|1.4235162 |
|22     |11      |1     |1424380312|1.2958367 |
|22     |13      |1     |1424380312|1.6077822 |
|22     |14      |1     |1424380312|1.32439   |
|22     |16      |1     |1424380312|0.6993503 |
|22     |18      |3     |1424380312|3.0975258 |
|22     |19      |1     |1424380312|1.3423778 |
|22     |22      |5     |1424380312|4.0774937 |
|22     |25      |1     |1424380312|1.0064191 |
|22     |26      |1     |1424380312|1.1788158 |
|22     |29      |3     |1424380312|3.1672745 |
|22     |30      |5     |1424380312|3.9120502 |
|22     |32      |4     |1424380312|3.01

## Test ML Model

In [27]:
# Evaluate the Recommendation System
from pyspark.ml.evaluation import RegressionEvaluator
# Set up evaluator to compute RMSE
evaluator = RegressionEvaluator(
    metricName="rmse", 
    labelCol="rating", 
    predictionCol="prediction"
)

# Calculate RMSE
rmse = evaluator.evaluate(predictions)
print(f"Root-mean-square error (RMSE) = {rmse}")

Root-mean-square error (RMSE) = 0.5747881498423372


In [28]:
su.spark.stop()